## Lesson Overview

**What this lesson teaches:** how to split a source document into useful retrieval units—the first data-preparation step in a retrieval-augmented generation (RAG) pipeline.

**What's happening under the hood:**
1. Load a Markdown report as one long string.
2. Split it by characters, sentences, or document sections.
3. Add overlap where useful so context is not lost at chunk boundaries.
4. Compare the number, size, and readability of the resulting chunks.
5. Decide which strategy best preserves the information a retriever will need later.

The pattern to internalize: chunking is a trade-off. Small chunks can be precise but lose context; large chunks preserve context but may include irrelevant material and consume more model tokens.

# Lesson 11: Chunking Documents for RAG

A RAG system first retrieves relevant source material and then gives that material to a language model. Retrieval usually works on **chunks**, not entire documents, so chunk quality directly affects what the model gets to see.

This notebook compares three approachable strategies. Run the cells in order, inspect where boundaries fall, and predict the trade-offs before reading each comparison.

## The RAG Pipeline

```text
Source document → chunks → embeddings/index → retrieval → prompt context → model answer
                    ^
              this lesson
```

Chunking happens before embedding and retrieval. A boundary in the wrong place can separate a fact from its explanation; useful overlap can preserve that connection. Too much overlap, however, creates duplicates and increases storage and prompt cost.

## Load the Example Document

The path check lets this cell work whether the notebook kernel starts in the project root or inside `Claude_API_Training`. We use UTF-8 explicitly so document loading is predictable across systems.

In [1]:
from pathlib import Path
import re

report_candidates = (Path('report.md'), Path('Claude_API_Training/report.md'))
report_path = next((path for path in report_candidates if path.exists()), None)
if report_path is None:
    raise FileNotFoundError('Could not find report.md. Start the notebook from the project root or Claude_API_Training.')

text = report_path.read_text(encoding='utf-8')
print(f'Loaded {report_path}: {len(text):,} characters')
print(text[:300] + '...')

Loaded report.md: 18,305 characters
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of...


## Strategy 1: Fixed-Character Chunks

This strategy is simple and produces chunks with predictable maximum sizes. The overlap repeats the end of one chunk at the start of the next.

Its weakness is that characters do not understand language: a boundary can cut through a word, sentence, Markdown heading, or related idea. The validation also matters—if overlap is at least as large as the chunk, the loop would never advance.

In [2]:
def chunk_by_char(text, chunk_size=500, chunk_overlap=50):
    if chunk_size <= 0:
        raise ValueError('chunk_size must be greater than 0')
    if not 0 <= chunk_overlap < chunk_size:
        raise ValueError('chunk_overlap must be at least 0 and smaller than chunk_size')

    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])

        if end_idx == len(text):
            break
        start_idx = end_idx - chunk_overlap

    return chunks

In [3]:
char_chunks = chunk_by_char(text)
print(f'{len(char_chunks)} character chunks')
print(repr(char_chunks[0][-80:]))
print('--- boundary ---')
print(repr(char_chunks[1][:130]))

# The first 50 characters of chunk 2 should repeat the end of chunk 1.
assert char_chunks[0][-50:] == char_chunks[1][:50]

41 character chunks
"oundaries. This year's review highlights significant progress in ten critical ar"
--- boundary ---
'highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yiel'


## Strategy 2: Sentence Chunks

Sentence chunking keeps complete sentences together, which usually makes chunks more readable and semantically coherent. Here, overlap is measured in sentences rather than characters.

The regular expression is intentionally lightweight. Production text may contain abbreviations, decimals, tables, or unusual punctuation, so mature pipelines often use a dedicated sentence tokenizer.

In [4]:
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    if max_sentences_per_chunk <= 0:
        raise ValueError('max_sentences_per_chunk must be greater than 0')
    if not 0 <= overlap_sentences < max_sentences_per_chunk:
        raise ValueError('overlap_sentences must be at least 0 and smaller than the chunk size')

    sentences = [
        sentence.strip()
        for sentence in re.split(r'(?<=[.!?])\s+', text.strip())
        if sentence.strip()
    ]
    step = max_sentences_per_chunk - overlap_sentences

    return [
        ' '.join(sentences[start:start + max_sentences_per_chunk])
        for start in range(0, len(sentences), step)
    ]

In [5]:
sentence_chunks = chunk_by_sentence(text)
print(f'{len(sentence_chunks)} sentence chunks')
print(sentence_chunks[0])
print('\n--- NEXT CHUNK ---\n')
print(sentence_chunks[1])

33 sentence chunks
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`).

--- NEXT CHUNK ---

Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007

## Strategy 3: Markdown Section Chunks

The report already contains useful structure in its `##` headings. Splitting at those boundaries keeps each heading with its content, so the chunk carries a strong topic label. This is an example of **structure-aware** or **semantic** chunking.

Unlike the earlier version, the lookahead pattern does not consume the heading marker. Each resulting section therefore retains its original Markdown heading.

In [6]:
def chunk_by_section(document_text):
    return [
        section.strip()
        for section in re.split(r'(?=^##\s)', document_text, flags=re.MULTILINE)
        if section.strip()
    ]

In [7]:
section_chunks = chunk_by_section(text)
print(f'{len(section_chunks)} section chunks')
for index, chunk in enumerate(section_chunks):
    heading = chunk.splitlines()[0]
    print(f'{index:>2}: {len(chunk):>4} chars | {heading}')

15 section chunks
 0:   69 chars | # **Annual Interdisciplinary Research Review: Cross-Domain Insights**
 1: 1962 chars | ## Executive Summary
 2:  856 chars | ## Table of Contents
 3: 1137 chars | ## Methodology
 4: 1180 chars | ## Section 1: Medical Research - Understanding XDR-471 Syndrome
 5: 1245 chars | ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
 6: 1172 chars | ## Section 3: Financial Analysis - Q3 Performance and Outlook
 7: 1183 chars | ## Section 4: Scientific Experimentation - Characterization of Material Composite XT-5
 8: 1225 chars | ## Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts
 9: 1144 chars | ## Section 6: Product Engineering - Finalizing Model Zircon-5 Specifications
10: 1232 chars | ## Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)
11: 1175 chars | ## Section 8: Project Management - Progress on Project Cerberus Phase 2B
12: 1159 chars | ## Section 9: Pharmaceutical Devel

## Compare the Strategies

Chunk count and size are not quality scores, but they expose the trade-offs. Look especially at the minimum and maximum sizes. A very large section may need a second splitting pass; a very small chunk may lack enough context to retrieve well.

In [8]:
def describe_chunks(name, chunks):
    sizes = [len(chunk) for chunk in chunks]
    return {
        'strategy': name,
        'count': len(chunks),
        'min_chars': min(sizes),
        'avg_chars': round(sum(sizes) / len(sizes)),
        'max_chars': max(sizes),
    }

for summary in (
    describe_chunks('characters', char_chunks),
    describe_chunks('sentences', sentence_chunks),
    describe_chunks('sections', section_chunks),
):
    print(summary)

{'strategy': 'characters', 'count': 41, 'min_chars': 305, 'avg_chars': 495, 'max_chars': 500}
{'strategy': 'sentences', 'count': 33, 'min_chars': 237, 'avg_chars': 701, 'max_chars': 1023}
{'strategy': 'sections', 'count': 15, 'min_chars': 69, 'avg_chars': 1218, 'max_chars': 2187}


## Local Sanity Checks

These checks catch easy-to-miss errors before chunks are embedded or stored. They confirm that no strategy creates empty chunks, fixed-character overlap is correct, and section headings survive the split.

In [9]:
assert all(chunk.strip() for chunk in char_chunks)
assert all(chunk.strip() for chunk in sentence_chunks)
assert all(chunk.strip() for chunk in section_chunks)
assert char_chunks[0][-50:] == char_chunks[1][:50]
assert any(chunk.startswith('## Section 10:') for chunk in section_chunks)

try:
    chunk_by_char('example', chunk_size=5, chunk_overlap=5)
    raise AssertionError('Invalid overlap should have raised ValueError')
except ValueError:
    pass

print('All chunking checks passed.')

All chunking checks passed.


## Practice: Predict, Change, Inspect

Try these one at a time and predict the result before running: 

1. **Boundary experiment:** change character chunks to `chunk_size=250, chunk_overlap=0`. Find a word or sentence cut across two chunks.
2. **Overlap experiment:** compare sentence overlap values `0`, `1`, and `2`. What information is duplicated, and when is that useful?
3. **Retrieval thought experiment:** search the chunks for `INC-2023-Q4-011`. Which strategy returns the cleanest context, and which returns the most complete context?
4. **Hybrid challenge:** first split by section, then split only sections longer than 1,000 characters into sentence chunks. Preserve the section heading as metadata or prepend it to each child chunk.

Reflection questions:
- Why can smaller chunks improve retrieval precision?
- Why can overlap improve answer quality but increase cost and duplication?
- What document structure can you preserve instead of discarding?
- How would you test chunk quality using the questions your application must answer?

## Summary

- Character chunking is simple and size-predictable, but it ignores language boundaries.
- Sentence chunking preserves readable units, though sentence detection is imperfect.
- Section chunking preserves document meaning and useful heading context, but section sizes can vary widely.
- Overlap reduces context loss at boundaries while adding duplication, storage, and token cost.
- Input validation prevents chunking loops from stalling on invalid overlap settings.
- The best strategy depends on document structure and the questions retrieval must support.

The next RAG step is usually to attach metadata to these chunks, convert them into embeddings, and store them in an index for similarity search.